# Machine Learning aplicado a procesos empresariales — Clase 2
## Problemas empresariales y primer modelo en Python (Regresión)

**Objetivo de la sesión (práctica):**
1. Reconocer tipos de problemas (predicción, clasificación, segmentación, anomalías).
2. Preparar un dataset empresarial sencillo.
3. Entrenar un primer modelo de regresión.
4. Interpretar resultados (error y variables con mayor impacto).


## 1. Tipos de problemas (mapa mental rápido)

- **Predicción (Regresión):** la variable objetivo es numérica (ventas, costo, demanda).
- **Clasificación:** la variable objetivo es categórica (abandona/no abandona, fraude/no fraude).
- **Segmentación (Clustering):** agrupar entidades similares sin variable objetivo.
- **Anomalías:** detectar comportamientos raros (fraude, sensores defectuosos).

En este laboratorio vamos a trabajar **Predicción de ventas** (regresión).


## 2. Preparación del entorno
Ejecuta esta celda para importar librerías.

In [1]:
# --- Librerías para manejo de datos ---
import pandas as pd          # Para trabajar con tablas (DataFrames)
import numpy as np            # Para operaciones matemáticas (raíz cuadrada, etc.)

# --- Librerías de Machine Learning (scikit-learn) ---
from sklearn.model_selection import train_test_split   # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer          # Aplicar transformaciones distintas a cada columna
from sklearn.preprocessing import OneHotEncoder        # Convertir categorías (texto) a números (0/1)
from sklearn.pipeline import Pipeline                  # Encadenar pasos de preprocesamiento + modelo
from sklearn.linear_model import LinearRegression      # El modelo de regresión lineal
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Métricas de error

## 3. Cargar el dataset
Usaremos un dataset sintético con variables típicas de negocio.

In [2]:
# Leemos el archivo CSV y lo cargamos en un DataFrame (tabla)
df = pd.read_csv("ventas_ml_clase2.csv")

# Mostramos las primeras 5 filas para verificar que se cargó correctamente
df.head()

,marketing,precio,temporada,tienda,canal,ventas
0,5548.49,25.74,2,Sur,Tienda,461.70
1,3128.03,31.60,3,Occidente,Tienda,229.12
2,6350.81,37.94,3,Centro,Tienda,397.16
3,6693.02,34.28,3,Norte,Tienda,458.31
4,1488.14,30.45,1,Occidente,Tienda,197.70


## 4. Exploración rápida (EDA)
Verifica forma, tipos, valores faltantes y estadísticas básicas.

In [3]:
# ¿Cuántas filas y columnas tiene nuestro dataset?
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# ¿Qué tipo de dato tiene cada columna?
# float64 = número decimal, int64 = número entero, object = texto/categoría
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:12s} → {dtype}")

Filas: 1200, Columnas: 6

Tipos de datos:
  marketing    → float64
  precio       → float64
  temporada    → int64
  tienda       → object
  canal        → object
  ventas       → float64


In [4]:
# Verificamos si hay valores faltantes (nulos) en alguna columna
# Los valores nulos pueden causar errores en el modelo si no se tratan
nulos = df.isna().sum()
print("Valores faltantes por columna:")
print(nulos.to_string())
print()

# Mensaje según el resultado
if nulos.sum() == 0:
    print("El dataset está completo: no hay valores nulos en ninguna columna.")
else:
    print(f"ATENCIÓN: hay {nulos.sum()} valores nulos en total. Requiere tratamiento.")

Valores faltantes por columna:
marketing    0
precio       0
temporada    0
tienda       0
canal        0
ventas       0

El dataset está completo: no hay valores nulos en ninguna columna.


In [5]:
# Estadísticas descriptivas de TODAS las columnas (numéricas y categóricas)
# Para numéricas: muestra promedio (mean), desviación estándar (std), mínimo, máximo, percentiles
# Para categóricas: muestra cantidad de valores únicos (unique), el más frecuente (top) y su frecuencia (freq)
df.describe(include="all").transpose().head(12)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
marketing,1200.0,NaN,NaN,NaN,4961.787167,1768.35789,0.0,3772.115,5006.215,6107.785,10721.94
precio,1200.0,NaN,NaN,NaN,34.560042,7.154758,13.55,29.915,34.91,39.245,55.4
temporada,1200.0,NaN,NaN,NaN,2.5125,1.112449,1.0,2.0,3.0,3.0,4.0
tienda,1200,5,Norte,269,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal,1200,3,Tienda,702,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ventas,1200.0,NaN,NaN,NaN,383.720658,80.045432,91.45,331.1675,383.635,436.72,597.72


**Lectura de las estadísticas descriptivas:**
- **marketing:** inversión promedio de ~$4,962 con rango de $0 a $10,722. Hay alta variabilidad (std $1,768).
- **precio:** promedio ~$34.56, rango de $13.55 a $55.40. Distribución razonablemente centrada.
- **temporada:** valores de 1 a 4 (trimestres del año), con mediana en 3 (más datos en temporadas altas).
- **tienda:** 5 ubicaciones, la más frecuente es "Norte" (269 registros). Distribución relativamente balanceada.
- **canal:** 3 canales de venta, "Tienda" es el más frecuente (702 de 1200).
- **ventas (variable objetivo):** promedio ~$383.72, desviación estándar de $80.05. Rango de $91.45 a $597.72.

## 5. Definir problema de negocio
**Pregunta:** ¿podemos predecir las ventas usando variables como marketing, precio, temporada, tienda y canal?

- Variables predictoras (**X**): marketing, precio, temporada, tienda, canal  
- Variable objetivo (**y**): ventas


In [6]:
# --- Paso 1: Separar variables predictoras (X) de la variable objetivo (y) ---
# X = las columnas que el modelo usará para hacer predicciones
# y = la columna que queremos predecir (ventas)
X = df[["marketing", "precio", "temporada", "tienda", "canal"]]
y = df["ventas"]

# --- Paso 2: Dividir en datos de entrenamiento (80%) y prueba (20%) ---
# El modelo APRENDE con los datos de entrenamiento
# y lo EVALUAMOS con los datos de prueba (que nunca vio antes)
# random_state=42 fija la semilla para que el resultado sea reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas ({X_train.shape[0]/len(df)*100:.0f}% del total)")
print(f"Datos de prueba:        {X_test.shape[0]} filas ({X_test.shape[0]/len(df)*100:.0f}% del total)")
print()
print("El modelo aprenderá con el 80% de los datos y se evaluará con el 20% restante.")
print("Esto evita que el modelo 'memorice' los datos y permite medir su capacidad real de predicción.")

Datos de entrenamiento: 960 filas (80% del total)
Datos de prueba:        240 filas (20% del total)

El modelo aprenderá con el 80% de los datos y se evaluará con el 20% restante.
Esto evita que el modelo 'memorice' los datos y permite medir su capacidad real de predicción.


## 6. Pipeline de preparación + modelo
Como tenemos variables categóricas (tienda, canal), hacemos **One-Hot Encoding** dentro de un pipeline.
Esto es una práctica estándar en proyectos reales.

One-Hot Encoding es una técnica de preprocesamiento en aprendizaje automático que convierte variables categóricas en un formato numérico binario (0 o 1)


In [ ]:
# --- Definimos qué columnas son numéricas y cuáles son categóricas ---
numeric_features = ["marketing", "precio", "temporada"]    # Ya son números, no necesitan transformación
categorical_features = ["tienda", "canal"]                  # Son texto, hay que convertirlas a números

# --- Preprocesamiento ---
# ColumnTransformer aplica transformaciones DISTINTAS según el tipo de columna:
#   - "passthrough" = dejar las numéricas tal como están
#   - OneHotEncoder = convertir cada categoría en columnas de 0 y 1
#     Ejemplo: tienda="Norte" → tienda_Norte=1, tienda_Sur=0, tienda_Centro=0, ...
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# --- Modelo ---
# LinearRegression = regresión lineal (el modelo más simple de ML)
# Busca la mejor fórmula: ventas = b0 + b1*marketing + b2*precio + ...
model = LinearRegression()

# --- Pipeline ---
# Un pipeline encadena los pasos en orden: primero preprocesa, luego entrena el modelo
# Ventaja: se ejecuta todo en una sola línea y evita errores de orden
pipe = Pipeline(steps=[
    ("preprocess", preprocess),   # Paso 1: transformar los datos
    ("model", model)              # Paso 2: aplicar el modelo de regresión
])

pipe

## 7. Entrenar el modelo
Entrenamos en el set de entrenamiento.

In [ ]:
# Entrenar el modelo = hacer que aprenda los patrones de los datos de entrenamiento
# .fit() recibe los datos de entrada (X_train) y las respuestas correctas (y_train)
# Internamente calcula los coeficientes (pesos) que mejor predicen las ventas
pipe.fit(X_train, y_train)

## 8. Evaluación básica
Usaremos métricas estándar para regresión:

- **MAE** (Error Absoluto Medio): "en promedio, me equivoco por $X". Es la métrica más fácil de interpretar.
- **RMSE** (Raíz del Error Cuadrático Medio): también mide el error promedio, pero le da **más peso a las equivocaciones grandes**. Si el modelo se equivoca mucho en unas pocas predicciones, el RMSE sube más que el MAE. Por eso siempre RMSE ≥ MAE. Si ambos son parecidos, los errores son parejos; si el RMSE es mucho mayor, hay algunos errores muy grandes.
- **R²** (R cuadrado): proporción de la variabilidad de las ventas que el modelo logra explicar. Va de 0 a 1 (0% a 100%). Ejemplo: R² = 0.66 → el modelo explica el 66% de las diferencias en ventas.

In [ ]:
# --- Generar predicciones con los datos de prueba ---
# .predict() usa la fórmula aprendida para predecir ventas a partir de X_test
y_pred = pipe.predict(X_test)

# --- Calcular las 3 métricas de evaluación ---
mae = mean_absolute_error(y_test, y_pred)                # Error Absoluto Medio
rmse = np.sqrt(mean_squared_error(y_test, y_pred))       # Raíz del Error Cuadrático Medio
r2 = r2_score(y_test, y_pred)                            # R² (variabilidad explicada)

# Promedio de ventas reales (para dar contexto al error)
media_ventas = y_test.mean()

# --- Mostrar resultados ---
print("=" * 50)
print("         MÉTRICAS DE EVALUACIÓN")
print("=" * 50)
print(f"  MAE  (Error Absoluto Medio):   ${mae:.2f}")
print(f"  RMSE (Raíz del Error Cuadr.):  ${rmse:.2f}")
print(f"  R²   (Variabilidad explicada):  {r2:.4f} ({r2*100:.1f}%)")
print("=" * 50)
print()
print(f"  Media de ventas reales: ${media_ventas:.2f}")
print(f"  Error relativo (MAE/Media): {mae/media_ventas*100:.1f}%")
print()
print("INTERPRETACIÓN:")
print(f"  - En promedio, el modelo se equivoca en ${mae:.2f} por predicción.")
print(f"  - Esto representa un {mae/media_ventas*100:.1f}% respecto a la venta promedio.")
print(f"  - El modelo explica el {r2*100:.1f}% de la variabilidad de las ventas.")
print(f"  - El {(1-r2)*100:.1f}% restante se debe a factores no incluidos en el modelo.")
print()
if r2 >= 0.8:
    print("  VALORACIÓN: Buen ajuste para un modelo lineal.")
elif r2 >= 0.5:
    print("  VALORACIÓN: Ajuste moderado. Hay espacio para mejorar con modelos")
    print("  más complejos (Random Forest, Gradient Boosting) o agregando más variables.")
else:
    print("  VALORACIÓN: Ajuste bajo. Se recomienda revisar las variables o probar otros modelos.")

## 9. Coeficientes del modelo: ¿qué variables pesan más?

Un modelo de regresión lineal es básicamente una **fórmula de suma ponderada**:

> ventas = base + (peso1 x marketing) + (peso2 x precio) + (peso3 x temporada) + ...

Cada **coeficiente** (o peso) nos dice: **"si esta variable sube en 1 unidad, ¿cuánto suben o bajan las ventas?"**, asumiendo que todo lo demás se mantiene igual.

**Esto NO es lo mismo que una correlación.** La diferencia clave:
- **Correlación** = mide si dos variables se mueven juntas (sin considerar las demás variables).
- **Coeficiente** = mide el efecto de UNA variable sobre las ventas, **ya tomando en cuenta todas las demás variables del modelo**.

**Cómo leer la tabla de abajo:**
- Coeficiente **positivo** (+) = esa variable **aumenta** las ventas.
- Coeficiente **negativo** (-) = esa variable **reduce** las ventas.
- Coeficiente **cercano a cero** = esa variable casi no afecta las ventas.

# --- Extraer los coeficientes (pesos) que el modelo aprendió ---

# Obtenemos los nombres de las columnas categóricas después del OneHotEncoding
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_cols = ohe.get_feature_names_out(["tienda", "canal"]).tolist()

# Lista completa de variables: numéricas + las generadas por OneHotEncoder
feature_names = numeric_features + cat_cols

# Los coeficientes son los "pesos" de la fórmula:
# ventas = intercepto + coef1*marketing + coef2*precio + ...
coef = pipe.named_steps["model"].coef_

# Organizamos en una tabla ordenada de mayor a menor impacto
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coef
}).sort_values("coef", ascending=False)

# --- Mostrar la tabla de coeficientes ---
print("=" * 55)
print("       COEFICIENTES DEL MODELO (ordenados)")
print("=" * 55)
for _, row in coef_df.iterrows():
    signo = "+" if row["coef"] > 0 else ""
    barra = "▲" if row["coef"] > 0 else "▼"    # ▲ sube ventas, ▼ baja ventas
    print(f"  {barra} {row['feature']:20s}  {signo}{row['coef']:.4f}")
print("=" * 55)

**Interpretación de los coeficientes:**

**Variables que MÁS aumentan las ventas (coeficiente positivo):**
- **tienda_Norte (+47.13):** Las tiendas del Norte venden en promedio ~$47 más. Es la ubicación más rentable.
- **temporada (+36.74):** Por cada unidad que sube la temporada (ej. de T1 a T2), las ventas aumentan ~$37. La estacionalidad es un factor clave.
- **canal_Mayorista (+36.20):** Vender por mayorista genera ~$36 más que otros canales.

**Variables que REDUCEN las ventas (coeficiente negativo):**
- **tienda_Oriente (-40.38):** La zona con peor desempeño, ~$40 menos en ventas.
- **canal_Tienda (-36.25):** La venta en tienda física es el canal menos rentable.
- **precio (-2.08):** Por cada $1 de aumento en precio, las ventas bajan ~$2 (elasticidad del precio).

**Variable con impacto bajo:**
- **marketing (+0.016):** Cada $1 adicional en marketing genera solo ~$0.016 en ventas. El efecto individual es bajo, pero se acumula con inversiones de miles de dólares.

In [ ]:
# Mostramos la tabla completa de coeficientes como DataFrame
# (misma información que arriba, pero en formato de tabla para copiar/exportar)
coef_df.tail(15)

# --- Simulación "What-If" (¿Qué pasaría si...?) ---
# Tomamos UNA fila real del set de prueba como ejemplo
# y vamos a ver qué predice el modelo si cambiamos una variable
row = X_test.iloc[[0]].copy()
print("Datos de la observación seleccionada:")
print(row.to_string(index=False))

In [ ]:
# --- Predicción BASE: ¿cuánto predice el modelo con los datos originales? ---
pred_base = float(pipe.predict(row)[0])

# --- Escenario alternativo: ¿qué pasa si aumentamos el marketing un 10%? ---
row_more_marketing = row.copy()
row_more_marketing["marketing"] = row_more_marketing["marketing"] * 1.10  # +10%

# Predicción con el marketing aumentado
pred_more_marketing = float(pipe.predict(row_more_marketing)[0])

# --- Calcular la diferencia ---
mkt_original = float(row["marketing"].values[0])
mkt_nuevo = float(row_more_marketing["marketing"].values[0])
diferencia = pred_more_marketing - pred_base

# --- Mostrar resultados de la simulación ---
print("=" * 55)
print("        SIMULACIÓN WHAT-IF: +10% Marketing")
print("=" * 55)
print(f"  Inversión marketing original:  ${mkt_original:,.2f}")
print(f"  Inversión marketing (+10%):    ${mkt_nuevo:,.2f}")
print(f"  Aumento en inversión:          ${mkt_nuevo - mkt_original:,.2f}")
print("-" * 55)
print(f"  Predicción de ventas (base):   ${pred_base:,.2f}")
print(f"  Predicción de ventas (+10%):   ${pred_more_marketing:,.2f}")
print(f"  Incremento en ventas:          ${diferencia:,.2f}")
print("=" * 55)
print()
print("INTERPRETACIÓN:")
print(f"  Aumentar la inversión en marketing un 10% (${mkt_nuevo - mkt_original:,.2f})")
print(f"  genera un incremento estimado de ${diferencia:,.2f} en ventas.")
print(f"  El retorno es positivo pero modesto: por cada $1 extra en marketing,")
print(f"  se obtienen ~${diferencia / (mkt_nuevo - mkt_original):.3f} adicionales en ventas.")

In [ ]:
# Verificación rápida de los valores numéricos (base, +10%, diferencia)
pred_base = float(pipe.predict(row)[0])

row_more_marketing = row.copy()
row_more_marketing["marketing"] = row_more_marketing["marketing"] * 1.10

pred_more_marketing = float(pipe.predict(row_more_marketing)[0])

# Resultado: (predicción base, predicción con +10% mkt, diferencia en ventas)
pred_base, pred_more_marketing, (pred_more_marketing - pred_base)

## 11. Discusión guiada (negocio)
1. ¿El MAE (Error Absoluto) es aceptable para este contexto? ¿por qué?
2. ¿Qué variable parece tener mayor impacto?
3. ¿Qué riesgo ves en usar este modelo en una decisión real?


## 12. Actividad final (en equipos, 15–20 min)
1. Propongan un problema real de su sector.
2. Definan el tipo de problema (regresión / clasificación / segmentación / anomalías).
3. Definan variable objetivo y 5 variables predictoras.
4. Expliquen cómo medirían el éxito (métrica + impacto empresarial).

Entregable: 1 slide o 1 página (texto) por equipo.
